# Hemisphere selector

Run cells in order. Results are saved to `conf/hemisphere_params.json` and reused by all downstream figure scripts.

**Steps**
1. Lasso OIL hemisphere
2. Lasso CORT hemisphere
3. Adjust rotation / flip (edit values, re-run preview cell)
4. Save

> **Kernel**: select the `.venv` kernel (or whichever has `anndata`, `plotly`).

In [ ]:
import json
from pathlib import Path

import anndata as ad
import numpy as np
import plotly.graph_objects as go
import yaml
from scipy.spatial import ConvexHull

ROOT = Path(".").resolve()
CONFIG_PATH = ROOT / "conf" / "spatial_expression.yaml"
OUTPUT_PATH = ROOT / "conf" / "hemisphere_params.json"

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

H5AD_PATH = ROOT / cfg["h5ad_path"]

# Sample IDs — change here if you want different samples
OIL_ID  = "B14"
CORT_ID = "B03"

# Subsample size for display (Scattergl handles it, but keeps the browser snappy)
DISPLAY_N = 40_000
PREVIEW_N =  6_000

# ── helpers ──────────────────────────────────────────────────────────────────

def load_coords(h5ad_path, sample_id):
    adata = ad.read_h5ad(h5ad_path, backed="r")
    mask = (adata.obs["sample"] == sample_id).values
    xy = adata.obsm["X_spatial"][mask]
    return xy[:, 0].astype(float), xy[:, 1].astype(float)

def subsample(x, y, n):
    if len(x) > n:
        idx = np.random.default_rng(42).choice(len(x), n, replace=False)
        return x[idx], y[idx]
    return x, y

def hull_from_selection(trace, point_inds):
    """Convex hull of selected points → crop polygon."""
    xs = np.array(trace.x)[point_inds]
    ys = np.array(trace.y)[point_inds]
    pts = np.column_stack([xs, ys])
    hull = ConvexHull(pts)
    verts = pts[hull.vertices].tolist()
    verts.append(verts[0])   # close polygon
    return verts

def apply_hull(x, y, hull_verts):
    """Boolean mask: points inside the convex hull polygon."""
    from matplotlib.path import Path
    path = Path(hull_verts)
    return path.contains_points(np.column_stack([x, y]))

def rotate(x, y, deg):
    cx, cy = x.mean(), y.mean()
    rad = np.deg2rad(deg)
    c, s = np.cos(rad), np.sin(rad)
    return cx + c*(x-cx) - s*(y-cy), cy + s*(x-cx) + c*(y-cy)

def transform(x, y, rot_deg, flip_x):
    if flip_x:
        x = 2*x.mean() - x
    return rotate(x, y, rot_deg)

def lasso_scatter(x, y, title, sample_id):
    """Return a FigureWidget ready for lasso selection."""
    xd, yd = subsample(x, y, DISPLAY_N)
    fig = go.FigureWidget([
        go.Scattergl(
            x=xd, y=yd,
            mode="markers",
            marker=dict(size=2, color="lightgray", opacity=0.7),
            selected=dict(marker=dict(color="steelblue", size=3, opacity=1.0)),
            unselected=dict(marker=dict(opacity=0.3)),
            name=sample_id,
        )
    ])
    fig.layout.update(
        title=dict(text=title, font=dict(size=13)),
        dragmode="lasso",
        width=620, height=620,
        xaxis=dict(scaleanchor="y", constrain="domain", showticklabels=False),
        yaxis=dict(constrain="domain", showticklabels=False),
        plot_bgcolor="white",
        margin=dict(l=20, r=20, t=50, b=20),
    )
    return fig

# Storage for selections
sel = {"oil": {"hull_verts": None, "n_pts": 0},
       "cort": {"hull_verts": None, "n_pts": 0}}

print("Setup complete.")

---
## Step 1 — OIL hemisphere

1. Run the cell below — a scatter of all OIL beads will appear.
2. In the Plotly toolbar (top-right), click the **lasso icon**.
3. Draw around the hemisphere you want to keep.
4. Run the **Confirm OIL** cell below to lock in the selection.

In [ ]:
print(f"Loading OIL ({OIL_ID}) …")
x_oil, y_oil = load_coords(H5AD_PATH, OIL_ID)
print(f"  {len(x_oil):,} beads total")

fig_oil = lasso_scatter(x_oil, y_oil,
    title=f"OIL  ({OIL_ID}) — lasso the hemisphere to keep",
    sample_id=OIL_ID)
fig_oil

In [ ]:
# ── Confirm OIL selection ────────────────────────────────────────────────────
# Run this AFTER drawing your lasso on the scatter above.

point_inds = list(fig_oil.data[0].selectedpoints or [])
if len(point_inds) < 10:
    print("No lasso selection detected. Draw a lasso first, then re-run this cell.")
else:
    sel["oil"]["hull_verts"] = hull_from_selection(fig_oil.data[0], point_inds)
    sel["oil"]["n_pts"] = len(point_inds)

    # Show how many full-dataset beads fall inside the hull
    mask_oil = apply_hull(x_oil, y_oil, sel["oil"]["hull_verts"])
    print(f"OIL selection confirmed.")
    print(f"  Selected in display subset : {len(point_inds):,}")
    print(f"  Beads inside hull (full set): {mask_oil.sum():,} / {len(x_oil):,}")

    # Quick confirmation plot
    xd, yd = subsample(x_oil, y_oil, DISPLAY_N)
    mask_d  = apply_hull(xd, yd, sel["oil"]["hull_verts"])
    fig_chk = go.Figure([
        go.Scattergl(x=xd[~mask_d], y=yd[~mask_d], mode="markers",
                     marker=dict(size=2, color="lightgray"), name="excluded"),
        go.Scattergl(x=xd[mask_d],  y=yd[mask_d],  mode="markers",
                     marker=dict(size=2, color="steelblue"), name="selected"),
    ])
    fig_chk.layout.update(width=500, height=500,
        xaxis=dict(scaleanchor="y", showticklabels=False),
        yaxis=dict(showticklabels=False),
        plot_bgcolor="white", title="OIL — confirmed selection",
        margin=dict(l=20, r=20, t=40, b=20))
    fig_chk.show()

---
## Step 2 — CORT hemisphere

Same process: lasso the hemisphere, then run the Confirm cell.

In [ ]:
print(f"Loading CORT ({CORT_ID}) …")
x_cort, y_cort = load_coords(H5AD_PATH, CORT_ID)
print(f"  {len(x_cort):,} beads total")

fig_cort = lasso_scatter(x_cort, y_cort,
    title=f"CORT  ({CORT_ID}) — lasso the hemisphere to keep",
    sample_id=CORT_ID)
fig_cort

In [ ]:
point_inds = list(fig_cort.data[0].selectedpoints or [])
if len(point_inds) < 10:
    print("No lasso selection detected. Draw a lasso first, then re-run this cell.")
else:
    sel["cort"]["hull_verts"] = hull_from_selection(fig_cort.data[0], point_inds)
    sel["cort"]["n_pts"] = len(point_inds)

    mask_cort = apply_hull(x_cort, y_cort, sel["cort"]["hull_verts"])
    print(f"CORT selection confirmed.")
    print(f"  Selected in display subset : {len(point_inds):,}")
    print(f"  Beads inside hull (full set): {mask_cort.sum():,} / {len(x_cort):,}")

    xd, yd = subsample(x_cort, y_cort, DISPLAY_N)
    mask_d  = apply_hull(xd, yd, sel["cort"]["hull_verts"])
    fig_chk = go.Figure([
        go.Scattergl(x=xd[~mask_d], y=yd[~mask_d], mode="markers",
                     marker=dict(size=2, color="lightgray"), name="excluded"),
        go.Scattergl(x=xd[mask_d],  y=yd[mask_d],  mode="markers",
                     marker=dict(size=2, color="salmon"), name="selected"),
    ])
    fig_chk.layout.update(width=500, height=500,
        xaxis=dict(scaleanchor="y", showticklabels=False),
        yaxis=dict(showticklabels=False),
        plot_bgcolor="white", title="CORT — confirmed selection",
        margin=dict(l=20, r=20, t=40, b=20))
    fig_chk.show()

---
## Step 3 — Rotation & alignment

Edit the four values below, then **re-run the cell** to see the preview. Repeat until both hemispheres look right.

- **Rotation**: degrees counter-clockwise. Positive = CCW, negative = CW.
- **Flip X**: mirrors the hemisphere left-right (useful to make both face the same direction).
- **Gap**: spacing between the two panels in µm in the final figure.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# EDIT THESE VALUES, then re-run this cell to update the preview
OIL_ROTATION_DEG  =  0.0    # e.g. -15.0
OIL_FLIP_X        = False   # True to mirror OIL left-right
CORT_ROTATION_DEG =  0.0
CORT_FLIP_X       = False
X_GAP_UM          =  500    # gap between the two panels (µm)
# ═══════════════════════════════════════════════════════════════

# ── build preview (subsampled) ───────────────────────────────────────────────
assert sel["oil"]["hull_verts"]  is not None, "Run OIL confirm cell first."
assert sel["cort"]["hull_verts"] is not None, "Run CORT confirm cell first."

mask_oil  = apply_hull(x_oil,  y_oil,  sel["oil"]["hull_verts"])
mask_cort = apply_hull(x_cort, y_cort, sel["cort"]["hull_verts"])

xo, yo = subsample(x_oil[mask_oil],   y_oil[mask_oil],   PREVIEW_N)
xc, yc = subsample(x_cort[mask_cort], y_cort[mask_cort], PREVIEW_N)

xo_t, yo_t = transform(xo.copy(), yo.copy(), OIL_ROTATION_DEG,  OIL_FLIP_X)
xc_t, yc_t = transform(xc.copy(), yc.copy(), CORT_ROTATION_DEG, CORT_FLIP_X)

# Normalize each to [0, width] so we can place them side by side
def norm_to_origin(x, y):
    return x - x.min(), y - y.min()

xo_n, yo_n = norm_to_origin(xo_t, yo_t)
xc_n, yc_n = norm_to_origin(xc_t, yc_t)
# Shift CORT to the right of OIL
xc_n = xc_n + xo_n.max() + X_GAP_UM

fig_prev = go.Figure([
    go.Scattergl(x=xo_n, y=yo_n, mode="markers",
                 marker=dict(size=2, color="steelblue", opacity=0.7),
                 name=f"OIL ({OIL_ID})"),
    go.Scattergl(x=xc_n, y=yc_n, mode="markers",
                 marker=dict(size=2, color="salmon",    opacity=0.7),
                 name=f"CORT ({CORT_ID})"),
])
fig_prev.layout.update(
    title=f"Preview — OIL rot={OIL_ROTATION_DEG}° flip={OIL_FLIP_X} | "
          f"CORT rot={CORT_ROTATION_DEG}° flip={CORT_FLIP_X}",
    xaxis=dict(scaleanchor="y", showticklabels=False),
    yaxis=dict(showticklabels=False),
    plot_bgcolor="white",
    width=1000, height=520,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig_prev.show()

---
## Step 4 — Save

When the preview looks good, run the cell below. It writes `conf/hemisphere_params.json`.

All downstream figure scripts will load that file automatically.

In [ ]:
assert sel["oil"]["hull_verts"]  is not None, "OIL selection missing."
assert sel["cort"]["hull_verts"] is not None, "CORT selection missing."

params = {
    "oil": {
        "sample": OIL_ID,
        "hull_poly": sel["oil"]["hull_verts"],
        "rotation_deg": OIL_ROTATION_DEG,
        "flip_x": OIL_FLIP_X,
    },
    "cort": {
        "sample": CORT_ID,
        "hull_poly": sel["cort"]["hull_verts"],
        "rotation_deg": CORT_ROTATION_DEG,
        "flip_x": CORT_FLIP_X,
    },
    "alignment": {
        "x_gap_um": X_GAP_UM,
    },
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, "w") as f:
    json.dump(params, f, indent=2)

print(f"Saved → {OUTPUT_PATH}")
print(f"  OIL  hull: {len(sel['oil']['hull_verts'])} vertices, "
      f"rot={OIL_ROTATION_DEG}°, flip={OIL_FLIP_X}")
print(f"  CORT hull: {len(sel['cort']['hull_verts'])} vertices, "
      f"rot={CORT_ROTATION_DEG}°, flip={CORT_FLIP_X}")
print(f"  gap: {X_GAP_UM} µm")